# Known-signal toy control

This is the first gate for the representation-learning project. It does not use a VAE, a decoder, or a learned `PhiNetwork`. It evolves particles directly and asks whether measuring kernel distances in the known 2D signal coordinates improves convergence over raw 32D distances.

If the known representation does not help here, stop before designing a learned representation.

In [ ]:
from pathlib import Path

# Make the notebook work when launched from the repository root or this folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Could not find src/ from {Path.cwd()}")

import os
os.chdir(PROJECT_ROOT)

from src.diagnostics import plot_discrepancy_curves
from src.experiments import ExperimentConfig
from src.toy import run_toy_particle_comparison

## Fixed protocol

The two methods share the same target, initial particles, target-index schedule, temperature, update size, evaluation metrics, and seeds. The only changed factor is the representation used inside the kernel.

In [ ]:
RUN_CONFIG = ExperimentConfig(
    dataset="toy32",
    methods=("raw", "known_signal"),
    seeds=(7, 17, 27, 37, 47, 57),
    drift_steps=100,
    eval_every=5,
    primary_metric="signal_swd",
    epsilon=None,  # choose this only after seeing discovery-scale curves
    fixed_budget=100,
    extras={"save_artifacts": False},
)

TOY_KWARGS = dict(
    n_samples=512,
    ambient_dim=32,
    signal_dim=2,
    centers=8,
    cluster_std=0.6,
    nuisance_scale=1.0,
    initial_signal_scale=1.5,
    batch_size=256,
    n_landmarks=128,
    temperature=1.0,
    step_size=0.1,
    backend="xpress",
)

RUN_CONFIG, TOY_KWARGS

## Run the particle comparison

`run_toy_particle_comparison` creates the same toy problem and paired random schedule for every seed, evolves particles, and records checkpoint metrics.

In [ ]:
comparison = run_toy_particle_comparison(RUN_CONFIG, **TOY_KWARGS)
print("External evaluation summary (signal_swd):")
for row in comparison.summary:
    print(
        f"{row['method']}: final={row['final_mean']:.4f} +/- {row['final_std']:.4f}; "
        f"AUC={row['auc_mean']:.4f} +/- {row['auc_std']:.4f}"
    )
comparison.summary

## Convergence curves

The primary curve is `signal_swd`. The observed and nuisance curves are guardrails: a method should not improve the signal by damaging dimensions that were already matched.

These values come from the frozen `DistributionEvaluator` called by `EvolutionRecorder`; they are external SWD/MMD measurements, not the particle-update loss.

In [ ]:
for metric_name in ("signal_swd", "observed_swd", "nuisance_swd"):
    figure, axis = plot_discrepancy_curves(
        comparison,
        metric_name=metric_name,
        show_individual_seeds=True,
    )
    display(figure)

In [ ]:
# Discovery mode keeps the summary and curves in this notebook only.
comparison.summary

## System walkthrough: one seed and five Xpress updates

Before designing a learned representation, inspect one short run end to end. This section uses the same Xpress field as the comparison above, but exposes the objects that the comparison function normally keeps inside the loop.

**Hypothesis:** measuring kernel distances after removing nuisance coordinates will make DriftXpress converge faster than measuring distances in the raw coordinates. This is an oracle test: `known_signal` uses the true signal projection supplied by the toy-data generator, not a learned `phi`.

- **raw:** `r(x) = x`, so the kernel sees all 32 coordinates.
- **known_signal:** `r(x) = P_signal x`, so the kernel sees only the true 2-dimensional signal.

Both methods keep particles and fields in the original 32-dimensional coordinates. The representation changes only kernel distances. The Xpress field attracts particles toward the target and repels particles from one another; the resulting 32-dimensional field is added to the particles. The evaluator then measures the state externally.

In [ ]:
from src.data import make_paired_run_inputs, make_toy_drift_problem
from src.diagnostics import kernel_effective_neighbors
from src.evaluators import make_toy_evaluator
from src.field import apply_representation, build_xpress_cache, compute_xpress_field, exponential_distance_kernel
import torch

WALK_SEED = 7
WALK_STEPS = 5
WALK_TEMPERATURE = TOY_KWARGS["temperature"]
WALK_STEP_SIZE = TOY_KWARGS["step_size"]

walk_problem = make_toy_drift_problem(
    n_samples=TOY_KWARGS["n_samples"],
    ambient_dim=TOY_KWARGS["ambient_dim"],
    signal_dim=TOY_KWARGS["signal_dim"],
    centers=TOY_KWARGS["centers"],
    cluster_std=TOY_KWARGS["cluster_std"],
    nuisance_scale=TOY_KWARGS["nuisance_scale"],
    initial_signal_scale=TOY_KWARGS["initial_signal_scale"],
    random_state=WALK_SEED,
)
walk_inputs = make_paired_run_inputs(
    n_steps=WALK_STEPS,
    dataset_size=TOY_KWARGS["n_samples"],
    batch_size=TOY_KWARGS["batch_size"],
    noise_dim=1,
    n_landmarks=TOY_KWARGS["n_landmarks"],
    evaluation_size=1,
    seed=WALK_SEED,
)
walk_evaluator = make_toy_evaluator(walk_problem, metrics=("swd",))
walk_representations = {
    "raw": None,
    "known_signal": walk_problem.project_signal,
}
walk_particles = {name: walk_problem.initial.detach().clone() for name in walk_representations}
walk_caches = {
    name: build_xpress_cache(
        walk_problem.target,
        landmark_indices=walk_inputs.landmark_indices,
        temperature=WALK_TEMPERATURE,
        representation=representation,
    )
    for name, representation in walk_representations.items()
}

print("Same initial particles:", torch.equal(walk_particles["raw"], walk_particles["known_signal"]))
print("Shared Xpress landmark count:", walk_inputs.landmark_indices.numel())
print("First ten shared landmark indices:", walk_inputs.landmark_indices[:10].tolist())
print("Initial particle shape:", tuple(walk_problem.initial.shape))
print("First particle (first four coordinates):", walk_problem.initial[0, :4].tolist())

for name, representation in walk_representations.items():
    print('\n===== {} ====='.format(name))
    particles = walk_particles[name]
    cache = walk_caches[name]
    field = None
    update = None
    for step in range(WALK_STEPS + 1):
        represented = apply_representation(particles, representation)
        positive_kernel = exponential_distance_kernel(
            particles, cache.landmarks, temperature=WALK_TEMPERATURE, representation=representation
        )
        negative_kernel = exponential_distance_kernel(
            particles, particles, temperature=WALK_TEMPERATURE, representation=representation
        ).clone()
        negative_kernel.fill_diagonal_(0.0)
        positive_neighbors = kernel_effective_neighbors(positive_kernel)
        negative_neighbors = kernel_effective_neighbors(negative_kernel)
        metrics = walk_evaluator.evaluate(particles)
        mean_field_norm = float(field.norm(dim=1).mean()) if field is not None else float('nan')
        mean_update_norm = float(update.norm(dim=1).mean()) if update is not None else float('nan')
        print('step={:02d} | represented_dim={} | signal_swd={:.4f} | mean_field_norm={:.4f} | mean_update_norm={:.4f}'.format(step, represented.shape[1], metrics['signal_swd'], mean_field_norm, mean_update_norm))
        print('  positive kernel: min={:.3g}, max={:.3g}, effective landmarks={:.1f}'.format(positive_kernel.min().item(), positive_kernel.max().item(), positive_neighbors.mean_effective_neighbors))
        print('  negative kernel effective neighbors={:.1f}'.format(negative_neighbors.mean_effective_neighbors))
        if field is not None:
            print('  field[0, :4]=', field[0, :4].tolist())
            print('  update[0, :4]=', update[0, :4].tolist())
        print("  particle[0, :4]=", particles[0, :4].tolist())
        if step == WALK_STEPS:
            break
        with torch.no_grad():
            field = compute_xpress_field(
                particles,
                cache=cache,
                representation=representation,
            )
            update = WALK_STEP_SIZE * field
            particles = particles + update

    walk_particles[name] = particles

print("\nThe Xpress cache uses the first fixed landmark set above. The full six-seed comparison uses the same set for raw and known_signal; the exact backend is the path that consumes the per-step positive batches.")

### How to read the walkthrough

At each step, `signal_swd` is the external distance to the target. `represented_dim` should be 32 for `raw` and 2 for `known_signal`. The particle and field remain 32-dimensional in both cases.

The kernel summaries show the direct query-to-landmark and particle-to-particle weights used as inputs to Xpress (the target attraction then passes through its cached Nystrom quantities). The field norm shows the size of the proposed motion, while the field/update and particle printouts show that the update is applied in the original coordinates.

This walkthrough is not a new benchmark and does not train `phi`; it is a microscope for understanding the benchmark before we add a learned representation.